# Session 8 · **Is It Actually Any Good?**
## Evaluating RAG with RAGAS, and turning a pipeline into a system

---

## 0.1 · The question we've been avoiding

Five sessions ago we started building. Since then we've added hybrid search, reranking, multi-query, routing, sub-questions, four index types, memory, and guardrails-by-prompt.

Prasad asks a reasonable question: **"Is it better than what we had in Session 5?"**

And we don't know. We've never measured.

We know how to measure *retrieval* — Session 4 gave us Recall@K and Precision@K. But watch what those miss.

> **The scenario.** Your retriever pulls three excellent passages about ashwagandha coffee: its adaptogenic properties, its calming effect, its earthy taste. Recall@K is high. Precision@K is high. Then the model answers:
>
> *"Ashwagandha coffee tastes fruity, like berries."*
>
> **The retrieval metrics applaud. The user is misinformed.**

Recall@K measured the pipes *before* the model. It cannot see what the model did with what it was handed.

---

## 0.2 · Four questions retrieval metrics cannot answer

| Question | Metric that answers it |
|---|---|
| Did the model actually *use* the documents? | Context Utilization |
| Did it invent details that weren't in the context? | **Faithfulness** |
| Did the answer address what the user actually asked? | Answer Relevancy |
| Were the retrieved chunks meaningful, or just vaguely on-topic? | Context Relevancy |

These are what **RAGAS** measures. It looks at the whole triple — *question, retrieved context, generated answer* — instead of stopping at retrieval.

> **The framing to keep:** everything you've learned so far — `top_k`, `alpha`, rerankers, compression, prompts — are **knobs**. RAGAS is the **dashboard**. Without a dashboard you're turning knobs in the dark and calling it engineering.

---

## 0.3 · Today's plan

| Part | Topic |
|---|---|
| 1 | Setup and the pipeline we'll measure |
| 2 | How RAGAS thinks: the triple, and offline vs online |
| 3 | Online metrics — no ground truth needed |
| 4 | Offline metrics — and where ground truth comes from |
| 5 | **Reading the scores: which number blames which component** |
| 6 | Comparing two pipelines with evidence |
| 7 | The improvement loop |
| 8 | Guardrails: acting on a score in real time |
| 9 | Reliability: empty hits, stale indexes, unsafe answers |
| 10 | Skeleton vs system |

### 📘 About the code
Same rule as before: `for` loops, `if` statements, lists, dictionaries, f-strings. No lambdas, comprehensions, or classes.

### ⚠️ Two installation traps — verified this week
RAGAS is currently **broken on a plain install**. Both fixes are in Cell 1.

| Problem | Symptom | Fix |
|---|---|---|
| `ragas` imports a module that `langchain-community` 0.4 removed | `ModuleNotFoundError: langchain_community.chat_models.vertexai` on `import ragas` | pin `langchain-community<0.4` |
| `context_utilization` was removed from `ragas.metrics` | `ImportError: cannot import name 'context_utilization'` | use the class `ContextUtilization` instead |

<details>
<summary><b>Why the pin? (worth reading — this is a useful kind of bug to recognise)</b></summary>

`ragas` 0.4.3 declares its dependency as plain `langchain-community` with **no upper version bound**. So pip installs the newest one, 0.4.x. But `langchain-community` 0.4 deleted `chat_models/vertexai.py` during its retirement, and `ragas/llms/base.py` imports it on **line 12, unconditionally**:

```python
from langchain_community.chat_models.vertexai import ChatVertexAI   # not inside a try/except
```

Unbounded dependency + hard import of a module that dependency removed = `import ragas` fails before you write any code. It has nothing to do with Vertex AI — you hit it on Gemini too.

**Why not something else?**
- *Upgrade ragas* — 0.4.3 is the latest; no fix has shipped.
- *Skip langchain-community* — can't; ragas imports it, and our hybrid retriever lives there too.
- *Stub the missing module* — fragile, and a bad habit.

**Is the pin safe?** Checked: on 0.3.31 the hybrid retriever still imports, `langchain-core` stays at 1.6.2, and LangChain 1.x / LlamaIndex / `langchain-google-genai` all keep working.

**The honest caveat:** we are pinning to a package that is itself being retired. This is a workaround for someone else's bug, not a design choice. **Try removing the pin before you teach this again** — if `import ragas` works without it, delete it.

**The transferable lesson:** when a library fails on import, read the traceback's last line for the *module path*, then check whether your version of that package still contains it. Nine times out of ten it's an unbounded dependency that drifted.
</details>

### 💰 A note on cost
**Every RAGAS metric is itself LLM calls.** Evaluating 5 questions on 5 metrics can be 50+ calls. On a free Gemini tier you will hit rate limits. We keep eval sets to **3 questions** all session — that's deliberate, not laziness.


---
# PART 1 · Setup and the pipeline we'll measure

First the pipeline from Session 6, rebuilt compactly. This is the thing under test.

In [ ]:
!pip -q install "langchain-community<0.4" ragas langchain-core langchain-google-genai langchain-pinecone pinecone pinecone-text beautifulsoup4

import os
import glob
import time

os.environ["GOOGLE_API_KEY"] = open("gemini_key2.txt").read().strip()
os.environ["PINECONE_API_KEY"] = open("api_key.txt").read().strip()

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from bs4 import BeautifulSoup

from pinecone import Pinecone

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

# --- match the embedding size to the index, instead of guessing it ---
# gemini-embedding-001 can return 768, 1536 or 3072 numbers.
# The index was built at ONE of those. Ask the index which, then match it.
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "coffee-hybrid"
index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()
index_dimension = stats["dimension"]

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=index_dimension
)

print("Index dimension:", index_dimension)
print("Embeddings set to match. Ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.6 MB/s eta 0

In [ ]:
def read_one_page(filename):
    html = open(filename, encoding="utf-8", errors="ignore").read()
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "head"]):
        tag.decompose()

    title = soup.find("h1").get_text(strip=True)
    page_id = os.path.basename(filename).replace(".html", "")

    documents = []
    for heading in soup.find_all("h2"):
        section_name = heading.get_text(strip=True)

        section_text = ""
        for item in heading.next_siblings:
            if getattr(item, "name", None) == "h2":
                break
            if hasattr(item, "get_text"):
                section_text = section_text + " " + item.get_text(" ", strip=True)
        section_text = " ".join(section_text.split())

        if len(section_text) > 40:
            info = {"page": page_id, "title": title, "section": section_name}
            full_text = title + " - " + section_name + ". " + section_text
            documents.append(Document(page_content=full_text, metadata=info))

    return documents


chunks = []
for filename in sorted(glob.glob("coffee_pages/*.html")):
    for document in read_one_page(filename):
        chunks.append(document)

print("Pieces:", len(chunks))

Pieces: 120


In [ ]:
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever

# pc and index were created in the setup cell, together with the matching embedder
texts = []
for chunk in chunks:
    texts.append(chunk.page_content)

bm25 = BM25Encoder().default()
bm25.fit(texts)

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, sparse_encoder=bm25, index=index, top_k=4, alpha=0.5
)

concierge_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the DilSeVector Concierge, a friendly coffee assistant.\n"
     "Answer ONLY from the CONTEXT below.\n"
     "If the CONTEXT does not answer the question, say you do not know. Never guess.\n"
     "Keep it under 80 words.\n\n"
     "CONTEXT:\n{context}"),
    ("human", "{question}")
])

concierge_chain = concierge_prompt | llm | StrOutputParser()


def format_context(documents):
    lines = []
    for document in documents:
        lines.append("[" + document.metadata["page"] + "] " + document.page_content)
    return "\n\n".join(lines)


def run_pipeline(question, how_many=4, alpha_value=0.5):
    """Returns the answer AND the context, because RAGAS needs both."""
    retriever.top_k = how_many
    retriever.alpha = alpha_value
    documents = retriever.invoke(question)

    context_list = []
    for document in documents:
        context_list.append(document.page_content)

    answer = concierge_chain.invoke({"context": format_context(documents),
                                     "question": question})
    return answer, context_list


answer, context = run_pipeline("Is turmeric coffee caffeine free?")
print("ANSWER:", answer)
print("\nCONTEXT pieces:", len(context))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


  0%|          | 0/120 [00:00<?, ?it/s]

ANSWER: Hello! I do not know if turmeric coffee is caffeine-free, as the provided context does not mention its caffeine content. It only lists hot brewed coffee as a key base ingredient.

CONTEXT pieces: 4


### 1.1 · Adding conversation history

A real Concierge is multi-turn. That changes evaluation in a way people miss.

> **The trap:** if turn 2 is *"compare it with masala"*, the RAGAS sample must record the **resolved** question, not the literal one. Otherwise answer relevancy scores the model against a question containing the word "it" — and punishes it for something the user never meant.
>
> **Rule: evaluate the question your retriever actually saw.**

In [ ]:
history = []


def format_history():
    if len(history) == 0:
        return "no previous conversation"
    lines = []
    for turn in history[-4:]:
        lines.append(turn[0] + ": " + turn[1])
    return "\n".join(lines)


chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the DilSeVector Concierge. Answer ONLY from the CONTEXT.\n"
     "If the CONTEXT does not answer it, say you do not know.\n\n"
     "CONVERSATION SO FAR:\n{history}\n\n"
     "CONTEXT:\n{context}"),
    ("human", "{question}")
])

chat_chain = chat_prompt | llm | StrOutputParser()


def run_chat(question, how_many=4):
    """Returns the answer, the context, AND the resolved question for evaluation."""
    resolved = question
    if len(history) >= 2:
        resolved = history[-2][1] + " " + question     # glue the last question on

    retriever.top_k = how_many
    documents = retriever.invoke(resolved)

    context_list = []
    for document in documents:
        context_list.append(document.page_content)

    answer = chat_chain.invoke({"history": format_history(),
                                "context": format_context(documents),
                                "question": question})
    history.append(["User", question])
    history.append(["Assistant", answer])
    return answer, context_list, resolved


answer_a, context_a, resolved_a = run_chat("Tell me about ashwagandha coffee.")
answer_b, context_b, resolved_b = run_chat("Compare it with turmeric coffee.")

print("turn 2 literal question :", "Compare it with turmeric coffee.")
print("turn 2 RESOLVED question:", resolved_b[:80], "...")
print("\nEvaluate with the resolved one, or answer relevancy will be unfairly low.")

turn 2 literal question : Compare it with turmeric coffee.
turn 2 RESOLVED question: Tell me about ashwagandha coffee. Compare it with turmeric coffee. ...

Evaluate with the resolved one, or answer relevancy will be unfairly low.


> **Notice what `run_pipeline` returns.** Not just the answer — the **context too**. Every RAGAS metric needs to see what the model was given, not only what it said. If your production pipeline throws away the retrieved context after answering, you cannot evaluate it. Log the context.


> ### ⚠️ If you see `Vector dimension X does not match the dimension of the index Y`
> This is the most common error when switching embedding models, and it is worth understanding rather than just fixing.
>
> A Pinecone index is created with a **fixed** dimension and can never change it. `gemini-embedding-001` can return **768, 1536 or 3072** numbers — 3072 by default. So if your index was built at 3072 and you ask for 768, every query is rejected.
>
> That is why the setup cell **reads the dimension off the index** instead of hardcoding a number. Hardcoding works until someone rebuilds the index, and then it fails confusingly.
>
> **The deeper trap, which does NOT raise an error:** a matching dimension does not mean matching *vectors*. If your index was filled with `text-embedding-004` and you query it with `gemini-embedding-001` at the same size, Pinecone happily returns results — they are just nonsense, because the two models put meaning in different places. Different embedding model means **re-index everything**, as we said back in Session 3.


---
# PART 2 · How RAGAS thinks

RAGAS looks at a **triple**:

```
        user_input          (what was asked)
        retrieved_contexts  (what we gave the model)
        response            (what the model said)
        reference           (the correct answer — only if we have it)
```

Different metrics compare different *pairs* of those. That's the whole mental model:

| Metric | Compares | Needs ground truth? |
|---|---|---|
| **Faithfulness** | response ↔ context | ❌ no |
| **Answer Relevancy** | response ↔ question | ❌ no |
| **Context Utilization** | context ↔ response | ❌ no |
| **Context Precision** | context ↔ reference | ✅ yes |
| **Context Recall** | context ↔ reference | ✅ yes |
| **Answer Correctness** | response ↔ reference | ✅ yes |

### Which splits the metrics into two families

**Online metrics** (no ground truth) — can run on live traffic, right now, as guardrails.
**Offline metrics** (need ground truth) — for testing, regression checks, A/B comparisons before you ship.

> **Every one of these is computed by an LLM.** Faithfulness, for instance, breaks your answer into individual claims and asks the judge whether each claim is supported by the context. That's why it costs calls — and why the judge model matters. A weak judge gives noisy scores.

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-3.5-flash"))
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("Judge ready. Note: the judge is a model too - it can be wrong.")

# NOTE: gemini-3.x models use fixed sampling and IGNORE temperature, so we
# do not pass it (passing it just prints a warning on every call).
# The practical consequence: the judge is NOT fully deterministic. The same
# answer can score slightly differently across runs, so treat small score
# differences as noise, not as evidence.

Judge ready. Note: the judge is a model too - it can be wrong.


/tmp/ipykernel_1424/2835401889.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-3.5-flash"))
/tmp/ipykernel_1424/2835401889.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(embeddings)


### ⚠️ 2.1 · Settings that decide whether this works at all

Run RAGAS with its defaults on a free Gemini tier and **every score comes back `NaN`**. Here is exactly why:

| RAGAS default | Value | What it does to you |
|---|---|---|
| `max_workers` | **16** | fires 16 requests at once → free tier returns 429 rate-limit errors |
| `max_retries` | 10 | each failure retries, backing off… |
| `max_wait` | 60 | …up to 60 seconds a time |
| `timeout` | **180** | the job is killed at 180s, score becomes `NaN` |

So the retries never finish before the timeout, and you get this — which means *nothing was measured*:

```
Evaluating: 100% 9/9 [03:00<00:00, 180.02s/it]
ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
{'faithfulness': nan, 'answer_relevancy': nan, 'context_utilization': nan}
```

> **`180.02s/it` is the tell.** When every item takes *exactly* the timeout value, you are not measuring slowness — you are measuring the timeout.

**The fix: stop hammering the API.** Fewer workers, a longer ceiling. Slower, but it actually returns numbers.

In [ ]:
from ragas.run_config import RunConfig

# tuned for a free Gemini tier - raise max_workers if you have a paid quota
safe_config = RunConfig(
    max_workers=2,      # default is 16 - that is what triggers rate limiting
    timeout=600,        # default is 180 - too short once retries start
    max_retries=5,
    max_wait=30
)


def run_eval(dataset, metrics):
    """One place to change evaluation settings for the whole notebook."""
    return evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_embeddings,
        run_config=safe_config
    )


print("Evaluation settings: max_workers =", safe_config.max_workers,
      "| timeout =", safe_config.timeout, "seconds")

Evaluation settings: max_workers = 2 | timeout = 600 seconds


### 2.2 · Smoke test before you spend anything

Before evaluating a whole set, check the judge works on **one sample with one metric**. If this returns `NaN`, stop — nothing further will work, and you'll waste ten minutes discovering that.

🎲 **What a healthy result looks like:** a number between 0 and 1. Anything else means the judge is failing, not that your pipeline is bad.

In [ ]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.evaluation import evaluate
from ragas.metrics import Faithfulness

smoke_answer, smoke_context = run_pipeline("Is turmeric coffee caffeine free?")

smoke_sample = SingleTurnSample(
    user_input="Is turmeric coffee caffeine free?",
    response=smoke_answer,
    retrieved_contexts=smoke_context
)

smoke_result = run_eval(EvaluationDataset(samples=[smoke_sample]), [Faithfulness()])
print("\nsmoke test:", smoke_result)

score_value = smoke_result.to_pandas()["faithfulness"][0]
if score_value != score_value:
    print("\n>>> NaN. The judge is failing, not your pipeline.")
    print(">>> Try max_workers=1, check your API quota, or switch judge model.")
else:
    print("\n>>> Judge is working. Safe to evaluate the full set.")

/tmp/ipykernel_1424/342646821.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]


smoke test: {'faithfulness': 0.6667}

>>> Judge is working. Safe to evaluate the full set.


---
# PART 3 · Online metrics — no ground truth needed

Start here, because these work on questions you've never seen before. That makes them usable as live guardrails.

## 3.1 · Faithfulness — did the answer stick to the documents?

**How it works:** the judge splits the answer into individual claims, then checks each one against the retrieved context.

```
faithfulness = claims supported by context / total claims
```

`1.0` = every claim is grounded. Low = the model invented things. **This is the hallucination detector.**

We'll build three samples on purpose: one honest answer, one where we force a hallucination, and one normal question.

In [ ]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.evaluation import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy, ContextUtilization

question_one = "Is turmeric coffee caffeine free?"
answer_one, context_one = run_pipeline(question_one)

question_two = "What does ashwagandha coffee taste like?"
answer_two, context_two = run_pipeline(question_two)

# a deliberately ungrounded answer, using the SAME real context
fake_answer = "Ashwagandha coffee tastes fruity, like berries, and contains no caffeine at all."

samples = []
samples.append(SingleTurnSample(user_input=question_one, response=answer_one,
                                retrieved_contexts=context_one))
samples.append(SingleTurnSample(user_input=question_two, response=answer_two,
                                retrieved_contexts=context_two))
samples.append(SingleTurnSample(user_input=question_two, response=fake_answer,
                                retrieved_contexts=context_two))

online_dataset = EvaluationDataset(samples=samples)
print("Built", len(online_dataset), "samples (the third one is a planted hallucination)")

/tmp/ipykernel_1424/2610137263.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, ContextUtilization
/tmp/ipykernel_1424/2610137263.py:3: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, ContextUtilization
/tmp/ipykernel_1424/2610137263.py:3: DeprecationWarning: Importing ContextUtilization from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextUtilization
  from ragas.metrics import Faithfulness, ResponseRelevancy, C

Built 3 samples (the third one is a planted hallucination)


🎲 **Before you run it:** sample 3 uses the *same real context* as sample 2, but the answer is invented. Which metric should collapse — faithfulness, or answer relevancy?

*(This takes a minute. Each metric is several LLM calls per sample.)*

In [ ]:
online_results = run_eval(online_dataset, [Faithfulness(), ResponseRelevancy(), ContextUtilization()])

print(online_results)
print()
scores_table = online_results.to_pandas()
print(scores_table[["user_input", "faithfulness", "answer_relevancy", "context_utilization"]].to_string())

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: GoogleRateLimitError(Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 4.178635463s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'Genera

{'faithfulness': nan, 'answer_relevancy': nan, 'context_utilization': nan}

                                 user_input  faithfulness  answer_relevancy  context_utilization
0         Is turmeric coffee caffeine free?           NaN               NaN                  NaN
1  What does ashwagandha coffee taste like?           NaN               NaN                  NaN
2  What does ashwagandha coffee taste like?           NaN               NaN                  NaN


### Reading that output

Row 3 should show **low faithfulness but decent answer relevancy.** That combination is the signature of *fluent hallucination* — the answer is on-topic and well written, and untrue.

That's precisely the failure Recall@K cannot see.

> ### ⚠️ `context_utilization` moved
> Older tutorials do `from ragas.metrics import context_utilization` (lowercase). That **no longer exists**. Use the class `ContextUtilization` as above. The lowercase instances for `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall` and `answer_correctness` still work, but using classes consistently is safer.

## 3.2 · The other two online metrics

**Answer Relevancy** — the judge generates questions *from your answer*, then measures how close they are to the real question. If your answer wanders off-topic, the generated questions won't match. Catches "fluent but off-topic."

**Context Utilization** — did the answer actually lean on what we fetched? Low utilization with a good answer means **you're retrieving too much**: paying to fetch and send passages the model ignores. It's your `top_k` telling you it's too high.


---
# PART 4 · Offline metrics — and the ground-truth problem

These need a **reference answer**: what a correct answer looks like. That's the expensive part of evaluation, and the part people skip.

### Where ground truth comes from

| Method | How | Trade-off |
|---|---|---|
| **Human annotation** | experts write reference answers | best quality, slowest, costliest |
| **From your documents** | take a section, write the Q&A it supports | cheap, honest for factual questions |
| **LLM-as-judge generation** | a strong model writes references from the docs | fast, but the judge's errors become "truth" |
| **Benchmark datasets** | public sets like MS MARCO, HotpotQA | free, may not match your domain |
| **Majority voting** | several models answer, take the consensus | weak but better than nothing |

We'll use method 2 — writing references straight from our coffee pages. For factual questions this is honest and takes minutes.

> **Be suspicious of method 3.** If the same model writes the reference *and* the answer, you're mostly measuring the model's self-consistency, not its correctness.

In [ ]:
gold_questions = [
    "What is the cold brew steeping time?",
    "Is turmeric coffee caffeine free?",
    "What is ashwagandha coffee made from?"
]

# references written by hand, from our own pages
gold_answers = [
    "Cold brew is steeped for 12 to 18 hours at around 4 degrees Celsius.",
    "Turmeric coffee is caffeine free when it is made with milk alone, without a coffee shot.",
    "Ashwagandha coffee is made by blending ashwagandha root powder with coffee and milk."
]

offline_samples = []
for position in range(len(gold_questions)):
    question = gold_questions[position]
    answer, context = run_pipeline(question)
    offline_samples.append(SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=context,
        reference=gold_answers[position]
    ))
    print("prepared:", question)

offline_dataset = EvaluationDataset(samples=offline_samples)
print("\nReady:", len(offline_dataset), "samples with references")

### The three offline metrics

| Metric | Asks | Low score blames |
|---|---|---|
| **Context Recall** | did we retrieve everything the reference needs? | the **retriever** — it missed facts |
| **Context Precision** | are the relevant chunks ranked near the top? | the **ranking** — noise is crowding out signal |
| **Answer Correctness** | is the final answer right? | the **whole pipeline** |

In [ ]:
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, AnswerCorrectness

offline_results = run_eval(offline_dataset, [LLMContextRecall(), LLMContextPrecisionWithReference(), AnswerCorrectness()])

print(offline_results)
print()
offline_table = offline_results.to_pandas()
columns = ["user_input", "context_recall", "llm_context_precision_with_reference", "answer_correctness"]
available = []
for name in columns:
    if name in offline_table.columns:
        available.append(name)
print(offline_table[available].to_string())

### 4.1 · Semantic Similarity — the cheap one

`AnswerCorrectness` is an LLM judge, so it costs calls and can be inconsistent. `SemanticSimilarity` just embeds both answers and compares vectors.

| | Answer Correctness | Semantic Similarity |
|---|---|---|
| How | LLM breaks both into claims and compares | cosine between two embeddings |
| Cost | LLM calls | embeddings only — cheap |
| Catches | factual disagreement | phrasing closeness |
| Misses | — | **a negation.** "is caffeine free" and "is not caffeine free" embed almost identically |

> **Use it as a fast regression check, never as your only correctness metric.** A high similarity score with a low correctness score usually means the answer is worded right and factually wrong — the most dangerous combination there is.

### Two more, if your references are detailed enough

Your notes list these as optional extensions, and they only make sense when your reference answers are rich:

- **Answer Coverage** — does the answer include every key aspect the reference contains? A reference listing four cautions and an answer giving two scores 0.5. Useful for checklist-style domains: compliance, safety, medical cautions.
- **Conciseness** — is the answer tight, or padded? Two answers can both be correct while one wastes half the context window.

Both need detailed references, which is why they're extensions rather than defaults. If your gold answers are one sentence, coverage is meaningless.


In [ ]:
from ragas.metrics import SemanticSimilarity

similarity_results = run_eval(offline_dataset, [SemanticSimilarity()])

print(similarity_results)
print("\nNo LLM calls for this metric - embeddings only.")

---
# PART 5 · 🔑 Reading the scores: which number blames which component

This table is the single most useful thing in the session. **Screenshot it.**

A RAG pipeline has three places to fail — retrieval, context preparation, generation. The *combination* of scores tells you which one.

| Context Recall | Faithfulness | Answer Relevancy | Diagnosis | Fix |
|---|---|---|---|---|
| 🔴 low | — | — | **Retriever missed it.** The facts never arrived. | ↑ `top_k`, adjust `alpha`, better chunking, multi-query |
| 🟢 high | 🔴 low | 🟢 high | **Model is inventing.** It had the facts and ignored them. | tighten the prompt, `temperature=0`, demand citations |
| 🟢 high | 🟢 high | 🔴 low | **Answering the wrong question.** Grounded but off-target. | improve the prompt, check query rewriting |
| 🟢 high | 🟢 high | 🟢 high | Working. | ship it |

And two more, using the other metrics:

| Signal | Diagnosis | Fix |
|---|---|---|
| Context Precision low, Recall high | right facts buried under noise | **add a reranker** — Session 4's fix |
| Context Utilization low | fetching far more than the model uses | **lower `top_k`** — you're paying for ignored tokens |

> **This is why RAGAS matters more than any single metric.** One number tells you something is wrong. The *pattern* tells you where.


In [ ]:
def diagnose(recall_score, faithfulness_score, relevancy_score):
    """Turn three numbers into a sentence about what to fix."""
    if recall_score < 0.6:
        return "RETRIEVER problem - the facts never arrived. Raise top_k or fix chunking."
    if faithfulness_score < 0.7:
        return "GENERATION problem - it had the facts and invented anyway. Tighten the prompt."
    if relevancy_score < 0.7:
        return "PROMPT problem - grounded, but answering the wrong question."
    return "Healthy."


print(diagnose(0.3, 0.9, 0.9))
print(diagnose(0.9, 0.4, 0.9))
print(diagnose(0.9, 0.9, 0.5))
print(diagnose(0.9, 0.9, 0.9))

---
# PART 6 · Comparing two pipelines with evidence

Now the payoff. We'll run the **same questions** through a deliberately weak pipeline and a stronger one, and let RAGAS settle the argument.

- **Pipeline A:** `top_k=1` — one passage only
- **Pipeline B:** `top_k=6` — more evidence

🎲 **Before you run it:** which metric will separate them most — faithfulness, or context recall?

In [ ]:
def build_samples(how_many, alpha_value=0.5):
    built = []
    for position in range(len(gold_questions)):
        question = gold_questions[position]
        answer, context = run_pipeline(question, how_many, alpha_value)
        built.append(SingleTurnSample(
            user_input=question,
            response=answer,
            retrieved_contexts=context,
            reference=gold_answers[position]
        ))
    return EvaluationDataset(samples=built)


metrics_to_use = [LLMContextRecall(), Faithfulness(), ResponseRelevancy()]

print("Evaluating Pipeline A (top_k = 1) ...")
result_a = run_eval(build_samples(1), metrics_to_use)

print("Evaluating Pipeline B (top_k = 6) ...")
result_b = run_eval(build_samples(6), metrics_to_use)

print("\nPipeline A (top_k=1):", result_a)
print("Pipeline B (top_k=6):", result_b)

> **What you're looking for.** Context recall should rise with `top_k`, because more passages means more chance the needed fact arrived. Faithfulness may *not* improve — and can even fall, since extra passages give the model more material to blend incorrectly.
>
> That trade-off is invisible to Recall@K. It's the whole reason the dashboard exists.


---
# PART 7 · The improvement loop

Evaluation is only useful if it changes something. The loop:

```
run pipeline → evaluate → read the diagnosis → change one knob → re-evaluate
```

**One knob at a time.** Change `top_k` and the prompt together and you'll never know which helped.

In [ ]:
def average_scores(evaluation_result):
    """Pull the mean of each metric out of a result object."""
    table = evaluation_result.to_pandas()
    summary = {}
    for column in table.columns:
        if column in ["user_input", "response", "retrieved_contexts", "reference"]:
            continue
        values = table[column].dropna()
        if len(values) > 0:
            summary[column] = round(float(values.mean()), 3)
    return summary


settings_to_try = [2, 5]
history = []

for k_value in settings_to_try:
    print("Trying top_k =", k_value, "...")
    result = run_eval(build_samples(k_value), [LLMContextRecall(), Faithfulness()])
    scores = average_scores(result)
    scores["top_k"] = k_value
    history.append(scores)

print("\nRESULTS")
for row in history:
    print("  ", row)

### 7.1 · Tuning two knobs together

One knob at a time is the rule when *diagnosing*. When *tuning*, you eventually need a small grid — because `top_k` and `alpha` interact. More passages (`top_k`↑) with pure keyword search (`alpha`=0) is not the same as more passages with pure semantic search.

Four combinations, three questions each. Keep it small — this is already 12 pipeline runs plus the evaluation calls.

In [ ]:
grid = [[2, 0.0], [2, 1.0], [6, 0.0], [6, 1.0]]
grid_history = []

for setting in grid:
    k_value = setting[0]
    alpha_value = setting[1]
    print("testing top_k =", k_value, " alpha =", alpha_value, "...")

    built = []
    for position in range(len(gold_questions)):
        question = gold_questions[position]
        answer, context = run_pipeline(question, k_value, alpha_value)
        built.append(SingleTurnSample(user_input=question, response=answer,
                                      retrieved_contexts=context,
                                      reference=gold_answers[position]))

    result = run_eval(EvaluationDataset(samples=built), [LLMContextRecall(), Faithfulness()])
    scores = average_scores(result)
    scores["top_k"] = k_value
    scores["alpha"] = alpha_value
    grid_history.append(scores)

print("\nGRID RESULTS")
for row in grid_history:
    print("  ", row)

> **What to look for.** `alpha = 0.0` is exact-words-only (Session 4's Aunty); `alpha = 1.0` is meaning-only. If recall jumps at `alpha = 0.0` for factual questions, your users' vocabulary matches your documents and you're over-weighting semantics.
>
> **And the reason to plot cost alongside:** `top_k = 6` may buy two points of recall for triple the prompt tokens. That's a business decision, not a technical one.

---
# PART 8 · Guardrails: acting on a score in real time

Offline metrics find problems *before* you ship. Online metrics can block a bad answer *as it happens*.

The pattern: **score the answer, and if it fails, don't send it.**

| Score | Threshold | Action |
|---|---|---|
| Faithfulness | < 0.7 | regenerate with stricter rules, or retrieve more |
| Faithfulness | still low after retry | refuse: "I don't have enough information" |
| Answer Relevancy | < 0.6 | re-run with a rewritten query |
| No documents retrieved | — | never call the model at all |

> **The rule from Session 5, now measurable:** a repair loop without a limit is an infinite loop. Two tries, then fall back honestly.
>
> **The cost, stated plainly:** checking faithfulness on every live answer doubles your LLM spend and adds seconds of latency. Real systems check a *sample* of traffic, or only high-stakes queries. Know what you're paying for.

In [ ]:
def score_faithfulness(question, answer, context):
    sample = SingleTurnSample(user_input=question, response=answer, retrieved_contexts=context)
    result = run_eval(EvaluationDataset(samples=[sample]), [Faithfulness()])
    table = result.to_pandas()
    value = table["faithfulness"][0]
    if value != value:        # a NaN means the judge could not decide
        return 0.0
    return float(value)


def guarded_answer(question):
    tries = 0
    how_many = 4

    while tries < 2:
        answer, context = run_pipeline(question, how_many)

        if len(context) == 0:                      # never call the model with nothing
            return "I could not find anything about that in our coffee guides."

        score = score_faithfulness(question, answer, context)
        print("   try", tries, "| top_k", how_many, "| faithfulness", round(score, 3))

        if score >= 0.7:
            return answer

        how_many = how_many * 2                    # repair: fetch more evidence
        tries = tries + 1

    return "I am not confident enough about that to answer. Could you rephrase the question?"


print("Q: Is turmeric coffee caffeine free?")
print("A:", guarded_answer("Is turmeric coffee caffeine free?"))

### 8.1 · Three kinds of guardrail

Your objectives name three, and they work at different points in the pipeline. Use all three — they catch different things.

| Kind | Where it acts | Catches | Cost |
|---|---|---|---|
| **Prompt-based** | before generation | most sloppiness — the "say you don't know" rule | free |
| **Structured parser** | at generation | wrong shape, missing citations, no refusal flag | free |
| **Validation chain** | after generation | ungrounded claims a rule can't express | LLM calls |

Prompt-based we've had since Session 5. Validation chains are Part 8's faithfulness check. The missing one is the **structured parser** — and it's the cheapest guardrail there is.

> **The idea:** instead of asking for prose and then inspecting it, ask the model for a **shape**. If it can't fill the shape, that itself is the signal.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

structured_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the DilSeVector Concierge. Use ONLY the CONTEXT.\n"
     'Reply with ONLY JSON like:\n'
     '{{"answer": "...", "pages_used": ["01_x"], "answered_from_context": true}}\n'
     "Set answered_from_context to false and answer to an empty string "
     "if the CONTEXT does not contain the answer.\n\n"
     "CONTEXT:\n{context}"),
    ("human", "{question}")
])

structured_chain = structured_prompt | llm | JsonOutputParser()


def structured_answer(question):
    documents = retriever.invoke(question)

    if len(documents) == 0:
        return "I don't know - nothing in our coffee guides covers that."

    result = structured_chain.invoke({"context": format_context(documents),
                                      "question": question})

    # guardrail A: the model flagged it itself
    if result.get("answered_from_context") is not True:
        return "I don't know - our coffee guides do not cover that."

    # guardrail B: no citations means nothing was grounded
    if len(result.get("pages_used", [])) == 0:
        return "I don't know - I could not point to a source for that."

    # guardrail C: cited a page that was never retrieved
    real_pages = []
    for document in documents:
        real_pages.append(document.metadata["page"])
    for page in result["pages_used"]:
        if page not in real_pages:
            return "I don't know - my sources did not check out."

    return result["answer"]


print("Q: What is the cold brew ratio?")
print("A:", structured_answer("What is the cold brew ratio?"))
print()
print("Q: What is the wifi password at the Dadar cafe?")
print("A:", structured_answer("What is the wifi password at the Dadar cafe?"))

> **Guardrail C is the one to notice.** The model handed us a list of pages it claims to have used, and we checked them against what was actually retrieved. That's Session 5's faithfulness check — for free, with no extra LLM call, because we asked for a shape instead of prose.

---
# PART 9 · Reliability: the three failures that hurt in production

Guardrails catch bad *answers*. These are bad *situations*.

| Situation | What happens without a plan | The plan |
|---|---|---|
| **Empty retrieval** | model answers from its own memory — hallucinating confidently | check the list is empty *before* calling the model; refuse |
| **Stale index** | confident answers from outdated documents | store an `updated_at` in metadata; surface the date; re-index on a schedule |
| **Unsafe generation** | medical or legal advice you didn't intend to give | topic check before answering; refuse-with-disclaimer path |

The first one is the most common and the easiest to fix — and almost nobody does it.

In [ ]:
UNSAFE_WORDS = ["pregnant", "pregnancy", "medication", "medicine", "dosage", "disease", "cure"]


def is_unsafe(question):
    lowered = question.lower()
    for word in UNSAFE_WORDS:
        if word in lowered:
            return True
    return False


def reliable_answer(question):
    # guard 1: unsafe topic
    if is_unsafe(question):
        return ("I can share what our guides say, but this sounds health related. "
                "Please check with a doctor. I am not able to give medical advice.")

    documents = retriever.invoke(question)

    # guard 2: nothing retrieved - do NOT call the model
    if len(documents) == 0:
        return "I could not find anything about that in our coffee guides."

    # guard 3: everything retrieved is weak
    best_score = 0.0
    for document in documents:
        score = document.metadata.get("score", 1.0)
        if score > best_score:
            best_score = score

    answer = concierge_chain.invoke({"context": format_context(documents),
                                     "question": question})
    return answer


for test_question in ["Is ashwagandha coffee safe during pregnancy?",
                      "What is the cold brew ratio?"]:
    print("Q:", test_question)
    print("A:", reliable_answer(test_question))
    print()

---
# PART 9.5 · Tracing: you cannot improve what you did not record

Every guardrail so far makes a decision and throws the evidence away. In production that's fatal — a user reports a bad answer next Tuesday and you have no idea what `k` was, which pages came back, or whether a repair fired.

**Log the decision, not just the answer.** Your notes list exactly what to capture:

| Field | Why |
|---|---|
| citations / pages used | lets you reproduce the answer |
| chosen `k` and `alpha` | the knobs that produced it |
| validator scores | why it passed or was repaired |
| outcome | `pass` / `repair` / `fallback` |
| latency and token counts | your SLO and your bill |

This is also how you build an eval set for free: **today's traces are next month's test cases.**

In [ ]:
trace_log = []


def traced_answer(question, how_many=4, alpha_value=0.5):
    started = time.time()

    retriever.top_k = how_many
    retriever.alpha = alpha_value
    documents = retriever.invoke(question)

    pages = []
    for document in documents:
        pages.append(document.metadata["page"])

    if len(documents) == 0:
        outcome = "fallback"
        answer = "I don't know - nothing in our coffee guides covers that."
    else:
        answer = concierge_chain.invoke({"context": format_context(documents),
                                         "question": question})
        outcome = "pass"

    record = {}
    record["question"] = question
    record["k"] = how_many
    record["alpha"] = alpha_value
    record["pages_used"] = pages
    record["outcome"] = outcome
    record["seconds"] = round(time.time() - started, 2)
    trace_log.append(record)

    return answer


traced_answer("What is the cold brew ratio?")
traced_answer("Who won the 1998 world cup?")

print("TRACE LOG")
for record in trace_log:
    print("  ", record)

---
# PART 10 · From skeleton to system

## 10.1 · The skeleton, one last time

Every pipeline in this course — Sessions 5, 6, 7 and today — has been the same five steps:

| Step | What it does | Knobs you now know |
|---|---|---|
| **Ingest** | load, clean, chunk, embed, store | chunk size, overlap, which embedder, metadata |
| **Retrieve** | candidate search over the index | `top_k`, `alpha`, filters, multi-query |
| **Prepare** | shortlist, dedupe, compress, budget | rerank, `top_n`, similarity cutoff |
| **Prompt** | shape the contract | role, grounding rule, citation rule, refusal |
| **Generate** | write the answer | model, `temperature`, response mode |

That's the **skeleton**: fixed steps, fixed knobs, one pass, no opinion about whether the output was any good.

## 10.2 · Decision axes — how you actually choose the knobs

Moving from skeleton to system means making these trade-offs *on purpose* instead of by default.

| Axis | Pulls one way | Pulls the other |
|---|---|---|
| **Latency vs accuracy vs cost** | more `top_k`, reranking, validators → better answers | a support bot with a 1.5s p95 budget cannot afford a cross-encoder plus three repair loops |
| **Open vs narrow domain** | open (Wikipedia-like) → broad queries, adaptive multi-query | narrow (internal policy) → hallucination is expensive → corrective loops mandatory |
| **Ground-truth availability** | have gold answers → offline metrics, regression tests | production only → online metrics as guardrails |
| **Failure tolerance** | FAQ bot can say "I don't know" | legal/medical **must** refuse when unsure |
| **Freshness vs stability** | daily documents → live ingestion, timestamp filters | stable corpus → tune offline once, index weekly |

> **Notice that none of these are answered by a metric.** RAGAS tells you what your system *is* doing. These axes tell you what it *should* be doing. You need both.

## 10.3 · Three systems, three different shapes

The same components, assembled differently for three real products.

### Flow 1 · Corrective RAG — policy, compliance, FAQ
**Rule: never hallucinate. Safer to refuse than to guess.**

```
retrieve → generate → score faithfulness
   ├─ ≥ 0.8            → send
   ├─ < 0.8 (1st time) → repair: "rewrite strictly from CONTEXT", re-check
   ├─ < 0.8 (2nd time) → widen evidence: top_k 8→15, alpha 0.4→0.3, regenerate
   └─ still failing    → refuse: "I don't know from the provided documents"
```
If **relevancy** is low but faithfulness is fine, don't touch `k` — that inflates recall and hides the problem. Reformulate the query instead, once.
**SLOs:** p95 ≤ 2.0s · hallucination rate < 1% · cost per query capped.
**Wire in:** feature flags (`corrective_on`, `use_reranker`), the trace log from Part 9.5, and a retrieval cache keyed on the normalised query.

### Flow 2 · Adaptive RAG — analyst and research assistants
**Rule: exploratory, comparative, multi-facet. Latency can be traded for coverage.**

```
classify the question
   ├─ narrow  → top_k 3, no rerank, compact
   └─ broad   → multi-query + sub-questions, top_k 12, rerank, tree_summarize
```
Session 5's flat-score signal still applies: if the retrieval scores come back flat, treat the question as hard regardless of how it looked.

### Flow 3 · Memory-RAG — personal assistants
**Rule: the user's own facts join the evidence — and that creates new ways to fail.**

```
retrieve documents  +  retrieve user memory  →  merge  →  generate with both cited
```
**Guardrails specific to memory:**
- **Never let memory override documents on facts.** A stored preference is not evidence about caffeine content.
- **Cite memory separately** from documents, so the user can see why they got this recommendation.
- **Stale preferences expire.** "Prefers low caffeine" from March may be wrong in September — timestamp it.
- **Deletion must be real.** If a user asks to be forgotten, their memory namespace has to be cleared, not just hidden.

---
# PART 10 · Skeleton vs system

Here's what changed today.

| | **Skeleton** (Sessions 5–7) | **System** (today) |
|---|---|---|
| Steps | fixed | adaptive — knobs change per query |
| Knobs | set once by guessing | set by **measurement** |
| Failure | returns something wrong | detects, repairs, or refuses |
| Loops | none — one pass | corrective loop with a budget |
| Empty retrieval | model invents | refuses before calling the model |
| Quality | "seems fine" | a number, tracked over time |
| Regression | noticed by a user | caught by an eval run |

> **The honest summary.** A skeleton is a pipeline that runs. A system is a pipeline that knows when it's failing and does something about it. The distance between them is measurement.

### What RAGAS does *not* do

- **It does not know your domain.** A judge model scoring coffee facts is fine. Scoring drug interactions is not.
- **The judge can be wrong.** Faithfulness returning `NaN` means the judge couldn't decide — treat that as a failure, not a pass. (Our `score_faithfulness` does exactly this.)
- **It is not free.** Every metric is LLM calls. Budget for evaluation like any other infrastructure.
- **Three samples is not a benchmark.** We used 3 to protect your rate limits. A real eval set is 50–200 questions covering the failures you actually care about.

> *"Session 5 taught you to build a RAG pipeline. Sessions 6 and 7 taught you to build it faster. Today you learned how to tell whether it works — and that's the one that makes you an engineer instead of a person who assembles components. Every knob you've learned to turn is useless until you can see the dial move."*

---

## Homework

1. **Build a real eval set.** Write 15 question/reference pairs from the coffee pages. Include 3 questions the pages *cannot* answer — the correct response is a refusal, and you should check the pipeline gives one.

2. **Find the trade-off.** Run `top_k` at 1, 3, 6 and 10 on your eval set. Plot context recall and faithfulness together. Find where recall stops improving but cost keeps rising.

3. **Tune alpha with evidence.** Same eval set, `alpha` at 0.0, 0.5 and 1.0. Which metric moves most? Connect it back to Session 4's α auction.

4. **Multi-turn evaluation.** Run a 3-turn conversation with `run_chat`. Build RAGAS samples using the **resolved** question, then again using the literal one. Compare answer relevancy and explain the gap.

5. **Break the judge.** Write an answer that is completely true but phrased very differently from the reference. See what answer correctness does. What does that tell you about trusting a single metric?

## Quick reference

| I want to… | Use |
|---|---|
| check for hallucination (no ground truth) | `Faithfulness()` |
| check the answer is on-topic | `ResponseRelevancy()` |
| check I'm not over-fetching | `ContextUtilization()` |
| check the retriever found the facts | `LLMContextRecall()` + reference |
| check ranking quality | `LLMContextPrecisionWithReference()` + reference |
| check the final answer is right | `AnswerCorrectness()` + reference |
| wrap a Gemini model as judge | `LangchainLLMWrapper(ChatGoogleGenerativeAI(...))` |
| build the eval data | `SingleTurnSample(...)` → `EvaluationDataset(samples=[...])` |
| run it | `evaluate(dataset=…, metrics=[…], llm=…, embeddings=…)` |
| see per-row scores | `result.to_pandas()` |
| cheap correctness check (no LLM) | `SemanticSimilarity()` |
| guardrail with zero extra cost | ask for JSON, check `pages_used` against what you retrieved |
| record why an answer happened | a trace dict: question, k, alpha, pages, outcome, seconds |


---

## Coverage check — the ten objectives

| # | Objective | Where |
|---|---|---|
| 1 | Why IR metrics aren't enough | Part 0 |
| 2 | Role of RAGAS: end-to-end, not just retrieval | Part 2 |
| 3 | Offline vs online metrics | Parts 3 and 4 |
| 4 | Implement RAGAS with Gemini as judge | Parts 3, 4 |
| 5 | Interpret scores to locate the failure | **Part 5 (the diagnosis table)** |
| 6 | Improvement strategies when metrics are weak | Parts 5, 6, 7 |
| 7 | Reliability fallbacks: empty hits, stale index, unsafe output | Part 9 |
| 8 | Guardrails: prompt, validation chain, structured parser | Parts 8, 8.1 |
| 9 | Feedback loop: run → evaluate → fix → re-evaluate | Parts 7, 7.1, 9.5 |
| 10 | Static skeleton vs dynamic system | Part 10 |
